# Day 4 | 최종 스크리닝
## Korean Stock Undervaluation Analysis — Final Screening

**목표:**
- EPS 수집 실패 종목 재수집 (계정명 확장)
- YoY 성장률 보완 (연간 EPS 2개년 대체)
- 3중 조건 스크리닝 → 최종 저평가 26개 확정
- gap_v2_final.csv 저장

**스크리닝 조건:**
```
① 시장 조정괴리율 ≤ -15%
② 섹터 조정괴리율 ≤ -15%
③ YoY > 0 또는 데이터 없음 (적자전환 제외)
④ 부채비율 < 200% (재무건전성)
```

**데이터 흐름:**
```
gap_v2.csv (136개)
    ↓
3중 조건 스크리닝
    ↓
gap_v2_final.csv (26개)
```


## 0. 환경 세팅

In [ ]:
!pip install -q git+https://github.com/FinanceData/FinanceDataReader.git

from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import requests
import pickle
import time
import os
from datetime import datetime

API_KEY   = "여기에_DART_API_키_입력"
BASE_PATH = '/content/drive/MyDrive/stock-analysis'

# 데이터 로드
df_main = pd.read_csv(
    f'{BASE_PATH}/data/processed/clean_data_final.csv',
    dtype={'Code': str, 'corp_code': str}
)
df_main['corp_code'] = df_main['corp_code'].str.zfill(8)

df_v2 = pd.read_csv(
    f'{BASE_PATH}/data/output/gap_v2.csv',
    dtype={'Code': str, 'corp_code': str}
)
df_v2['corp_code'] = df_v2['corp_code'].str.zfill(8)

print(f"✅ df_main: {len(df_main)}개")
print(f"✅ df_v2  : {len(df_v2)}개")
print(f"   YoY 있음: {df_v2['yoy_growth_pct'].notna().sum()}개")
print(f"   YoY 없음: {df_v2['yoy_growth_pct'].isna().sum()}개")


## 1. EPS 수집 실패 종목 재수집

In [ ]:
# Day 3에서 실패한 종목들 재수집
# 원인: 기업별 고유 EPS 계정명 (보통주/우선주 구분, 번호 접두사 등)

# pickle 체크포인트 로드 (Day 3에서 저장한 수집 데이터)
with open(f'{BASE_PATH}/data/temp/all_eps_checkpoint.pkl', 'rb') as f:
    all_eps = pickle.load(f)

print(f"체크포인트 로드: {len(all_eps)}개")

# 수집 안 된 종목 확인
missing = df_main[~df_main['Code'].isin(all_eps.keys())]
print(f"재수집 필요: {len(missing)}개")
print(missing[['Name', 'Code']].to_string(index=False))


## 2. 부채비율 계산 + 스크리닝

In [ ]:
# 부채비율 = 총부채 / 자기자본 × 100
df_v2['debt_ratio'] = (df_v2['total_liab'] / df_v2['equity'] * 100).round(1)

print("부채비율 분포:")
print(df_v2['debt_ratio'].describe())
print(f"\n200% 초과 종목: {(df_v2['debt_ratio'] >= 200).sum()}개")


## 3. 3중 조건 스크리닝

In [ ]:
# 스크리닝 조건 적용
# cond1: 두 조정괴리율 모두 -15% 이하 (시장·섹터 동시 저평가)
# cond2: YoY > 0 또는 데이터 없음 (적자전환 종목 제외)
# cond3: 부채비율 200% 미만 (재무건전성)

cond1 = (df_v2['gap_market_v2'] < -0.15) & (df_v2['gap_sector_v2'] < -0.15)
cond2 = (df_v2['yoy_growth_pct'] > 0) | (df_v2['yoy_growth_pct'].isna())
cond3 = (df_v2['debt_ratio'] < 200) | (df_v2['debt_ratio'].isna())

df_screened = df_v2[cond1 & cond2 & cond3].copy()
df_screened = df_screened.sort_values('gap_market_v2')

print("✅ 스크리닝 완료")
print(f"\n조건별 통과 종목:")
print(f"  cond1 (두 괴리율 모두 -15% 이하): {cond1.sum()}개")
print(f"  cond2 (YoY > 0 or 없음)         : {cond2.sum()}개")
print(f"  cond3 (부채비율 200% 이하)       : {cond3.sum()}개")
print(f"  최종 교집합                      : {len(df_screened)}개")
print(f"\n최종 저평가 종목:")
print(df_screened[['Name', 'gap_market_v2', 'gap_sector_v2',
                   'yoy_growth_pct', 'debt_ratio', 'signal_v2']].to_string(index=False))


## 4. 저장

In [ ]:
# 최종 스크리닝 결과 저장
df_screened.to_csv(
    f'{BASE_PATH}/data/output/gap_v2_final.csv',
    index=False, encoding='utf-8-sig'
)

# gap_v2 전체도 업데이트 저장 (부채비율 컬럼 추가)
df_v2.to_csv(
    f'{BASE_PATH}/data/output/gap_v2.csv',
    index=False, encoding='utf-8-sig'
)

print(f"💾 저장 완료!")
print(f"   gap_v2.csv      : {len(df_v2)}개 (전체)")
print(f"   gap_v2_final.csv: {len(df_screened)}개 (최종 스크리닝)")
print(f"\n최종 저평가 TOP 10:")
print(df_screened.head(10)[['Name', 'gap_market_v2', 'signal_v2']].to_string(index=False))
